# 29.07 - Class-imbalance training: weighted sampling

**Notebook type:** Solution.

**Daily output:** Sampler training comparison and a strategy note covering standard sampling, weighted loss, and `WeightedRandomSampler`.

You will inspect the distribution produced by a weighted sampler and compare three controlled training strategies on a noisy imbalanced image task.


## Core Ideas

- `WeightedRandomSampler` consumes one weight per sample, not one weight per class.
- Convert class weights to sample weights with `class_weights[train_labels]`.
- When a sampler is supplied, do not also set `shuffle=True`.
- `replacement=True` lets minority samples appear repeatedly; this changes the sampled training distribution and can overfit rare examples.
- Weighted loss keeps the observed batches natural but changes each example's loss contribution.
- Do not combine sampling and weighted loss blindly. Compare them independently first.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, recall_score

SEED = 29
NUM_CLASSES = 3
np.random.seed(SEED)
torch.manual_seed(SEED)


## Prepared Harder Imbalanced Image Data

The fixture uses shifted structures, distractors, occlusion, and severe class imbalance.

**Return structure — `make_imbalanced_pattern_images`:** Returns a tuple. Position 0 is a CPU `torch.float32` tensor `[N,3,24,24]` with values in `[0,1]`. Position 1 is a CPU `torch.long` tensor `[N]`. `N=sum(counts)` and labels are contiguous from zero.


In [ ]:
def make_imbalanced_pattern_images(counts=(75, 21, 12), image_size=24, seed=29):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for class_index, class_count in enumerate(counts):
        for sample_index in range(class_count):
            image = rng.normal(0.18, 0.22, size=(3, image_size, image_size)).astype(np.float32)
            shift = int(rng.integers(-3, 4))
            main_channel = int(rng.integers(0, 3))
            detail_channel = (main_channel + int(rng.integers(1, 3))) % 3
            if class_index in (0, 2):
                center = image_size // 2 + shift
                image[main_channel, 3:image_size - 3, center - 2:center + 2] += 0.52
                if class_index == 2:
                    # The rare class shares the majority vertical bar and differs
                    # only by a short, partially noisy cross-piece.
                    detail_row = image_size // 2 + int(rng.integers(-3, 4))
                    image[detail_channel, detail_row - 1:detail_row + 2, center - 6:center + 7] += 0.46
            elif class_index == 1:
                center = image_size // 2 + shift
                image[main_channel, center - 2:center + 2, 3:image_size - 3] += 0.52
            # Every class receives an unrelated patch and occasional occlusion.
            patch_top = int(rng.integers(2, image_size - 6))
            patch_left = int(rng.integers(2, image_size - 6))
            image[detail_channel, patch_top:patch_top + 4, patch_left:patch_left + 4] += 0.30
            if sample_index % 3 == 0:
                top = int(rng.integers(4, image_size - 8))
                left = int(rng.integers(4, image_size - 8))
                image[:, top:top + 5, left:left + 6] *= 0.08
            images.append(np.clip(image, 0.0, 1.0))
            labels.append(class_index)
    order = rng.permutation(len(labels))
    return torch.tensor(np.stack(images)[order], dtype=torch.float32), torch.tensor(np.asarray(labels)[order], dtype=torch.long)


images, labels = make_imbalanced_pattern_images()
print("dataset:", images.shape, images.dtype, torch.bincount(labels).tolist())


## Prepared Split and Model Helpers

These are supplied so the exercises remain focused on sampling strategy.

**Return structure — `prepare_day29_split`:** Returns a `dict` containing CPU tensors `train_images`, `val_images` (`torch.float32` rank 4), `train_labels`, `val_labels`, `train_indices`, and `val_indices` (`torch.long` rank 1). The split is stratified, disjoint, and normalized using training statistics only.


In [ ]:
def prepare_day29_split(images, labels, val_fraction=0.25, seed=SEED):
    indices = np.arange(len(labels))
    train_array, val_array = train_test_split(
        indices,
        test_size=val_fraction,
        random_state=seed,
        stratify=labels.numpy(),
    )
    train_indices = torch.tensor(train_array, dtype=torch.long)
    val_indices = torch.tensor(val_array, dtype=torch.long)
    raw_train = images[train_indices]
    raw_val = images[val_indices]
    mean = raw_train.mean()
    std = raw_train.std().clamp_min(1e-6)
    return {
        "train_images": ((raw_train - mean) / std).float(),
        "val_images": ((raw_val - mean) / std).float(),
        "train_labels": labels[train_indices].long(),
        "val_labels": labels[val_indices].long(),
        "train_indices": train_indices,
        "val_indices": val_indices,
    }


split = prepare_day29_split(images, labels)
print("train/validation counts:", torch.bincount(split["train_labels"]).tolist(), torch.bincount(split["val_labels"]).tolist())


**Return structure — `TinySamplerCNN`:** A callable `nn.Module`. Construction returns a CPU module. Calling it with a CPU `torch.float32` tensor `[N,3,H,W]` returns CPU `torch.float32` logits `[N,num_classes]`.


In [ ]:
class TinySamplerCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 12, 3, padding=1),
            nn.BatchNorm2d(12),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(12, 24, 3, padding=1),
            nn.BatchNorm2d(24),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(24, num_classes)

    def forward(self, batch):
        return self.classifier(self.features(batch).flatten(1))


print("prepared model output:", TinySamplerCNN(NUM_CLASSES)(split["train_images"][:3]).shape)


## Exercise 29-A: Build a weighted sampler

Calculate inverse-frequency class weights, map them to each training observation, and sample a balanced-length epoch with replacement.

**Return structure — `build_weighted_sampler`:** Returns a tuple. Position 0 is a `WeightedRandomSampler` with `replacement=True` and `num_samples=max_class_count*num_classes`. Position 1 is a CPU `torch.float64` tensor `[N_train]` containing one positive finite weight per training observation.


In [ ]:
def build_weighted_sampler(train_labels, num_classes, seed=SEED):
    counts = torch.bincount(train_labels, minlength=num_classes).double()
    if torch.any(counts == 0):
        raise ValueError("Every class must occur in the training split.")
    class_weights = counts.reciprocal()
    sample_weights = class_weights[train_labels]
    epoch_size = int(counts.max().item()) * num_classes
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=epoch_size,
        replacement=True,
        generator=torch.Generator().manual_seed(seed),
    )
    return sampler, sample_weights


# Smoke check
smoke_sampler, smoke_sample_weights = build_weighted_sampler(split["train_labels"], NUM_CLASSES)
print("sampler epoch size:", smoke_sampler.num_samples, "sample weights:", smoke_sample_weights.shape)


## Exercise 29-B: Audit the sampled distribution

Draw one sampler epoch and compare natural and sampled counts.

**Return structure — `audit_sampled_epoch`:** Returns a `dict` with `natural_counts` and `sampled_counts`, CPU `torch.long` tensors `[C]`; `sampled_indices`, a CPU `torch.long` tensor `[sampler.num_samples]`; and `max_min_ratio`, a finite positive Python `float` equal to maximum sampled count divided by minimum sampled count.


In [ ]:
def audit_sampled_epoch(train_labels, sampler, num_classes):
    sampled_indices = torch.tensor(list(sampler), dtype=torch.long)
    natural_counts = torch.bincount(train_labels, minlength=num_classes).long()
    sampled_counts = torch.bincount(train_labels[sampled_indices], minlength=num_classes).long()
    ratio = sampled_counts.max().item() / sampled_counts.min().item()
    return {
        "natural_counts": natural_counts,
        "sampled_counts": sampled_counts,
        "sampled_indices": sampled_indices,
        "max_min_ratio": float(ratio),
    }


# Smoke check
sampling_audit = audit_sampled_epoch(split["train_labels"], smoke_sampler, NUM_CLASSES)
print("natural/sampled:", sampling_audit["natural_counts"].tolist(), sampling_audit["sampled_counts"].tolist())


## Exercise 29-C: Train one imbalance strategy

Support exactly `standard`, `weighted_loss`, and `weighted_sampler`. Use only one intervention at a time.

**Return structure — `train_imbalance_strategy`:** Returns a `dict` with `strategy` (`str`), `history` (`list[float]` of length `epochs`), and `metrics`. `metrics` contains `accuracy`, `macro_f1` (Python floats), `per_class_recall` (CPU `torch.float32` tensor `[C]`), and `confusion_matrix` (CPU `torch.long` tensor `[C,C]`).


In [ ]:
def train_imbalance_strategy(split, num_classes, strategy, epochs=3, seed=SEED):
    if strategy not in {"standard", "weighted_loss", "weighted_sampler"}:
        raise ValueError("Unknown imbalance strategy.")
    torch.manual_seed(seed)
    model = TinySamplerCNN(num_classes)
    train_dataset = TensorDataset(split["train_images"], split["train_labels"])
    if strategy == "weighted_sampler":
        sampler, _ = build_weighted_sampler(split["train_labels"], num_classes, seed=seed)
        train_loader = DataLoader(train_dataset, batch_size=18, sampler=sampler)
    else:
        train_loader = DataLoader(
            train_dataset,
            batch_size=18,
            shuffle=True,
            generator=torch.Generator().manual_seed(seed),
        )
    counts = torch.bincount(split["train_labels"], minlength=num_classes).float()
    loss_weights = counts.sum() / (num_classes * counts)
    criterion = nn.CrossEntropyLoss(weight=loss_weights if strategy == "weighted_loss" else None)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.012)
    history = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        total_count = 0
        for batch_images, batch_labels in train_loader:
            optimizer.zero_grad()
            loss = criterion(model(batch_images), batch_labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch_labels)
            total_count += len(batch_labels)
        history.append(float(total_loss / total_count))

    model.eval()
    val_loader = DataLoader(TensorDataset(split["val_images"], split["val_labels"]), batch_size=18, shuffle=False)
    prediction_parts, label_parts = [], []
    with torch.inference_mode():
        for batch_images, batch_labels in val_loader:
            prediction_parts.append(model(batch_images).argmax(dim=1).cpu())
            label_parts.append(batch_labels.cpu())
    predictions = torch.cat(prediction_parts).long()
    validation_labels = torch.cat(label_parts).long()
    class_order = np.arange(num_classes)
    metrics = {
        "accuracy": float(accuracy_score(validation_labels.numpy(), predictions.numpy())),
        "macro_f1": float(f1_score(validation_labels.numpy(), predictions.numpy(), labels=class_order, average="macro", zero_division=0)),
        "per_class_recall": torch.tensor(recall_score(validation_labels.numpy(), predictions.numpy(), labels=class_order, average=None, zero_division=0), dtype=torch.float32),
        "confusion_matrix": torch.tensor(confusion_matrix(validation_labels.numpy(), predictions.numpy(), labels=class_order), dtype=torch.long),
    }
    return {"strategy": strategy, "history": history, "metrics": metrics}


# Smoke check
smoke_strategy = train_imbalance_strategy(split, NUM_CLASSES, "standard", epochs=1)
print("standard smoke Macro-F1:", smoke_strategy["metrics"]["macro_f1"])


## Exercise 29-D: Compare all three strategies

Run controlled experiments and retain their metrics without assuming that the most aggressive balancing method must win.

**Return structure — `compare_imbalance_strategies`:** Returns a `dict` with exactly `standard`, `weighted_loss`, and `weighted_sampler`. Each value is the matching dictionary from `train_imbalance_strategy` and uses the same requested epoch count.


In [ ]:
def compare_imbalance_strategies(split, num_classes, epochs=3, seed=SEED):
    return {
        strategy: train_imbalance_strategy(split, num_classes, strategy, epochs=epochs, seed=seed)
        for strategy in ("standard", "weighted_loss", "weighted_sampler")
    }


# Smoke check
strategy_results = compare_imbalance_strategies(split, NUM_CLASSES, epochs=3)
for name, result in strategy_results.items():
    print(name, "Macro-F1:", result["metrics"]["macro_f1"], "recall:", result["metrics"]["per_class_recall"].tolist())


## Exercise 29-E: Write a strategy decision

Choose the highest validation Macro-F1, then state when weighted sampling is risky.

**Return structure — `write_sampling_strategy_note`:** Returns a `dict` with exactly `selected_strategy` (`str`), `selection_reason` (`str`), `sampler_risk` (`str`), and `decision_rule` (`str`). Every string is non-empty and `selected_strategy` is one of the three compared keys.


In [ ]:
def write_sampling_strategy_note(results):
    selected = max(results, key=lambda name: results[name]["metrics"]["macro_f1"])
    score = results[selected]["metrics"]["macro_f1"]
    return {
        "selected_strategy": selected,
        "selection_reason": f"{selected} had the highest controlled validation Macro-F1 ({score:.3f}).",
        "sampler_risk": "Replacement can repeat rare observations many times and overfit minority-specific noise.",
        "decision_rule": "Compare baseline, weighted loss, and sampling separately; choose with untouched validation Macro-F1 and per-class recall.",
    }


# Smoke check
strategy_note = write_sampling_strategy_note(strategy_results)
print("selected:", strategy_note["selected_strategy"])
print("decision rule:", strategy_note["decision_rule"])


## Test Cases

Run this cell after completing all TODO cells. A correct implementation prints `Day 29 tests passed`.

**Return structure — `run_day29_tests`:** Returns `None`. Success is communicated by assertions completing and the exact printed message `Day 29 tests passed`.


In [ ]:
def run_day29_tests():
    sampler, sample_weights = build_weighted_sampler(split["train_labels"], NUM_CLASSES, seed=SEED)
    counts = torch.bincount(split["train_labels"], minlength=NUM_CLASSES)
    assert isinstance(sampler, WeightedRandomSampler) and sampler.replacement is True
    assert sampler.num_samples == int(counts.max().item()) * NUM_CLASSES
    assert sample_weights.shape == split["train_labels"].shape and sample_weights.dtype == torch.float64
    assert torch.allclose(sample_weights, counts.double().reciprocal()[split["train_labels"]])

    audit_sampler, _ = build_weighted_sampler(split["train_labels"], NUM_CLASSES, seed=SEED)
    audit = audit_sampled_epoch(split["train_labels"], audit_sampler, NUM_CLASSES)
    assert set(audit) == {"natural_counts", "sampled_counts", "sampled_indices", "max_min_ratio"}
    assert audit["sampled_indices"].shape == (sampler.num_samples,)
    assert int(audit["sampled_counts"].sum()) == sampler.num_samples
    assert audit["max_min_ratio"] < 2.0

    assert set(strategy_results) == {"standard", "weighted_loss", "weighted_sampler"}
    for name, result in strategy_results.items():
        assert result["strategy"] == name and len(result["history"]) == 3
        assert result["metrics"]["confusion_matrix"].shape == (NUM_CLASSES, NUM_CLASSES)
        assert int(result["metrics"]["confusion_matrix"].sum()) == len(split["val_labels"])
    note = write_sampling_strategy_note(strategy_results)
    assert set(note) == {"selected_strategy", "selection_reason", "sampler_risk", "decision_rule"}
    assert note["selected_strategy"] in strategy_results
    print("Day 29 tests passed")


run_day29_tests()


## Day 29 Checklist

- [ ] I converted class weights into per-sample weights.
- [ ] I used a sampler instead of `shuffle=True`.
- [ ] I inspected the sampled class distribution.
- [ ] I compared sampling and weighted loss independently.
- [ ] I can explain replacement, epoch size, minority overfitting, and when to prefer each strategy.
